# Original TVA bigram tokenizer trained on IAM

This notebook follows `train_tokenizers.ipynb`: prepare TVA annotations, call `get_tokenizer('bigram').train(...)`, and load the saved JSON. Run all cells with the **WSL TVA Python environment** from this repository.

The vocabulary is selected from the complete IAM training split only. It retains TVA's complete-label deduplication, unrestricted adjacent pairs (including spaces/punctuation), `Counter.most_common` ranking, and greedy left-to-right encoding. Raw IAM text is preserved: no lowercasing, stripping, candidate filtering, or extra normalization.

For the ED comparison, use **145 bigrams + 78 singleton characters + one CTC blank = 224 classes**. This matches the handwriting condition's size and alphabet; candidate eligibility differs. The custom IAM letter-frequency comparator remains a separate condition. This notebook trains a tokenizer; recognition-model training is a later manual step.


## 1. Imports and paths

Edit external paths here if needed. ED labels are read only in the final audit, after vocabulary selection and freezing. The ED alphabet is the already-declared task alphabet, not inferred from label frequencies.


In [1]:
from pathlib import Path
from collections import Counter
from itertools import combinations
import hashlib
import inspect
import json
import os
import pickle
import platform
import string
import subprocess
import sys
import tempfile

# Run from TVA's root, or a directory beneath it.
REPO_ROOT = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / 'tva/tokenizers.py').is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from tva.tokenizers import BigramTokenizer, get_tokenizer
from tva.ed_handwriting_bigram import ED_ALPHABET
# Pinned IAM source identities, independent of retired custom builders.
LABELS_SHA256 = '5ac34ad37ba0b125308fe1a2bc97095985e25dfa76495628c3bb3895c0b446ab'
SELECTION_SHA256 = '7893dfba4febe6df99cf0bdb0c74fabe7b736d2a6af05033d8638b90455bc1c2'

IAM_LABELS = Path('/home/artellisys/DTLR/data/IAM_new/labels.pkl')
IAM_SELECTION = Path('/home/artellisys/dtlr-output/iam-train-full/selection.json')
ED_DIRECTORY = Path(os.environ.get(
    'TVA_ED_DATASET_DIR', '/mnt/c/Users/Ali/Downloads/fau-english-dataset'
))
OUTPUT = REPO_ROOT / 'artifacts/tokenizers/ed_iam_tva_original_bigram_greedy_v1.json'
PROVENANCE = OUTPUT.with_suffix('.provenance.json')
AUDIT_OUTPUT = REPO_ROOT / 'artifacts/tokenizers/ed_iam_tva_original_bigram_greedy_v1.audit.json'
HANDWRITING = REPO_ROOT / 'artifacts/tokenizers/ed_iam_handwriting_bigram_greedy_v1.json'
CUSTOM_LINGUISTIC = REPO_ROOT / 'artifacts/tokenizers/ed_iam_linguistic_bigram_greedy_v1.json'
CATEGORIES = ['', *ED_ALPHABET]
BIGRAM_COUNT = 145
VOCAB_SIZE = len(CATEGORIES) + BIGRAM_COUNT
HASH_SEED = '42'
assert len(CATEGORIES) == 79 and VOCAB_SIZE == 224
print('Python:', sys.executable)
print('Repository:', REPO_ROOT)
print('Output:', OUTPUT)


Python: /home/artellisys/miniconda3/envs/tva/bin/python
Repository: /home/artellisys/TVA
Output: /home/artellisys/TVA/artifacts/tokenizers/ed_iam_tva_original_bigram_greedy_v1.json


## 2. Authenticate and read the IAM training split

The pinned hashes authenticate the existing labels and training manifest before loading the pickle. Check every line ID and transcription hash, and require exact coverage of all 5,694 training lines. IAM validation/test labels do not enter the training annotations.


In [2]:
def sha256(content):
    return hashlib.sha256(content).hexdigest()


def require(condition, message):
    if not condition:
        raise ValueError(message)


labels_bytes = IAM_LABELS.read_bytes()
selection_bytes = IAM_SELECTION.read_bytes()
require(sha256(labels_bytes) == LABELS_SHA256, 'IAM labels SHA-256 mismatch')
require(sha256(selection_bytes) == SELECTION_SHA256, 'IAM selection SHA-256 mismatch')
selection = json.loads(selection_bytes)
require(
    selection.get('schema_version') == 'dtlr.iam-selection.v1'
    and selection.get('dataset') == 'IAM'
    and selection.get('split') == 'train'
    and selection.get('requested_count') == 5694
    and selection.get('labels_sha256') == LABELS_SHA256,
    'Manifest must identify the complete IAM training split',
)
labels = pickle.loads(labels_bytes)
train_rows = labels['ground_truth']['train']
require(isinstance(train_rows, list) and len(train_rows) == 5694, 'Unexpected IAM train split')
by_id = {row['id']: row for row in train_rows}
require(len(by_id) == len(train_rows), 'Duplicate IAM training line ID')
selected_ids = [row['id'] for row in selection['lines']]
require(
    len(selected_ids) == 5694 and len(set(selected_ids)) == 5694
    and set(selected_ids) == set(by_id),
    'Manifest must cover every IAM training line exactly once',
)
annotations = []
for selected in selection['lines']:
    row = by_id[selected['id']]
    text = row['text']
    require(isinstance(text, str) and bool(text), 'Invalid IAM transcript')
    require(
        sha256(text.encode('utf-8')) == selected['transcription_sha256'],
        f'Transcription hash mismatch: {row["id"]}',
    )
    annotations.append({'id': row['id'], 'label': text})
unique_labels = {row['label'] for row in annotations}
print('IAM train lines:', len(annotations))
print('Unique complete labels counted by TVA:', len(unique_labels))
print('Repeated complete labels removed by TVA:', len(annotations) - len(unique_labels))


IAM train lines: 5694
Unique complete labels counted by TVA: 5337
Repeated complete labels removed by TVA: 357


## 3. Train with TVA's original implementation

All IAM training lines go into a temporary one-fold `train.json`. Fold `0` here denotes one source vocabulary, not an ED recognition fold. The output is shared across ED folds.

The central call is the same as TVA's notebook:

```python
tokenizer_bigram = get_tokenizer('bigram')
tokenizer_bigram.train(dir_ds, categories, 224)
```

TVA iterates `list(set(labels))`; tied counts inherit encounter order. A fresh Python process with `PYTHONHASHSEED=42` preserves this behavior reproducibly in the recorded Python version. Setting the variable inside a running notebook kernel would not reset its hash seed. No replacement selection algorithm is used.


In [3]:
training_code = """
import json
import sys
from tva.tokenizers import get_tokenizer

dir_ds, categories_json, size = sys.argv[1:]
categories = json.loads(categories_json)
tokenizer_bigram = get_tokenizer('bigram')
tokenizer_bigram.train(dir_ds, categories, int(size))
"""
training_env = dict(os.environ, PYTHONHASHSEED=HASH_SEED)
with tempfile.TemporaryDirectory(prefix='iam-tva-bigram-') as directory:
    dir_ds = Path(directory)
    train_json_bytes = json.dumps(
        {'info': {'num_fold': 1}, 'annotations': {'0': annotations}},
        ensure_ascii=False, indent=2,
    ).encode('utf-8')
    (dir_ds / 'train.json').write_bytes(train_json_bytes)
    result = subprocess.run(
        [sys.executable, '-c', training_code, str(dir_ds),
         json.dumps(CATEGORIES), str(VOCAB_SIZE)],
        cwd=REPO_ROOT, env=training_env, capture_output=True, text=True,
    )
    print(result.stdout, end='')
    print(result.stderr, end='')
    result.check_returncode()
    trained_bytes = (dir_ds / 'tokenizers' / f'bigram{VOCAB_SIZE}' / '0.json').read_bytes()
    # Re-run the original trainer to verify deterministic artifact bytes.
    repeated = subprocess.run(
        [sys.executable, '-c', training_code, str(dir_ds),
         json.dumps(CATEGORIES), str(VOCAB_SIZE)],
        cwd=REPO_ROOT, env=training_env, capture_output=True, text=True,
    )
    repeated.check_returncode()
    require(
        trained_bytes == (dir_ds / 'tokenizers' / f'bigram{VOCAB_SIZE}' / '0.json').read_bytes(),
        'Repeated original-TVA training produced different bytes',
    )
model = json.loads(trained_bytes)
vocab = model['vocab']
selected_bigrams = [token for token, index in sorted(vocab.items(), key=lambda item: item[1]) if len(token) == 2]
require(len(vocab) == VOCAB_SIZE, 'Unexpected output class count')
require(vocab.get('') == 0, 'CTC blank must have ID 0')
require({t for t in vocab if len(t) == 1} == set(ED_ALPHABET), 'Singleton alphabet mismatch')
require(len(selected_bigrams) == BIGRAM_COUNT, 'Expected 145 selected bigrams')
require(set(vocab.values()) == set(range(VOCAB_SIZE)), 'Invalid vocabulary IDs')
require(
    model['idx_token'] == {str(index): token for token, index in vocab.items()},
    'Forward and inverse mappings disagree',
)
require(
    all(char in ED_ALPHABET for token in selected_bigrams for char in token),
    'A selected unrestricted IAM pair falls outside the ED alphabet; do not silently filter it',
)
print('Classes:', len(vocab), '| bigrams:', len(selected_bigrams))
print('Repeated training: byte-identical')
print('Tokenizer SHA-256:', sha256(trained_bytes))


2026-10-03 22:58:47.020 | INFO     | tva.tokenizers:train:245 - BigramTokenizers are saved at /tmp/iam-tva-bigram-kos66hl9/tokenizers/bigram224.
Classes: 224 | bigrams: 145
Repeated training: byte-identical
Tokenizer SHA-256: 796a676bcfe4087acfdebf795a5e9797a8dd05ff93054d9e72de14e4d44eb73f


## 4. Freeze the native TVA JSON and record provenance

The artifact keeps the original `vocab`/`idx_token` format and loads with `tokenizer: bigram`. Provenance is stored in a sidecar. Existing files are accepted only when byte-identical; a differing frozen result raises an error. Use new versioned paths for a changed experiment.


In [4]:
def save_frozen(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_bytes() == content, f'Existing frozen file differs: {path}')
    else:
        with path.open('xb') as handle:
            handle.write(content)


def json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + '\n').encode('utf-8')


provenance = {
    'schema_version': 'tva.iam-original-bigram-provenance.v1',
    'condition': 'original-TVA-on-IAM linguistic bigram',
    'source_dataset': 'IAM', 'source_split': 'train', 'downstream_dataset': 'ED',
    'labels_sha256': LABELS_SHA256, 'selection_sha256': SELECTION_SHA256,
    'training_annotations_sha256': sha256(train_json_bytes),
    'line_count': len(annotations), 'unique_complete_label_count': len(unique_labels),
    'trainer': 'tva.tokenizers.BigramTokenizer.train',
    'trainer_source_sha256': sha256(inspect.getsource(BigramTokenizer.train).encode('utf-8')),
    'python_version': platform.python_version(), 'python_hash_seed': HASH_SEED,
    'selection': 'unique-complete-labels; all adjacent pairs; Counter.most_common',
    'text_preprocessing': 'none; raw IAM transcripts retained',
    'encoding': 'original-TVA greedy left-to-right',
    'categories': sorted(set(CATEGORIES)), 'size': VOCAB_SIZE, 'bigram_count': BIGRAM_COUNT,
    'blank_token': '', 'blank_id': 0,
    'ed_labels_read_for_selection': False,
    'repeated_training_byte_identical': True,
    'tokenizer_sha256': sha256(trained_bytes),
}
# Preflight both files before writing either one.
for path, content in [(OUTPUT, trained_bytes), (PROVENANCE, json_bytes(provenance))]:
    if path.exists():
        require(path.read_bytes() == content, f'Existing frozen file differs: {path}')
save_frozen(OUTPUT, trained_bytes)
save_frozen(PROVENANCE, json_bytes(provenance))
tokenizer_bigram = get_tokenizer('bigram')
tokenizer_bigram.load(str(OUTPUT))
require(tokenizer_bigram.size == 224, 'Saved tokenizer size mismatch')
print('Saved:', OUTPUT)
print('Provenance:', PROVENANCE)


Saved: /home/artellisys/TVA/artifacts/tokenizers/ed_iam_tva_original_bigram_greedy_v1.json
Provenance: /home/artellisys/TVA/artifacts/tokenizers/ed_iam_tva_original_bigram_greedy_v1.provenance.json


## 5. Compare vocabularies

Counts below are diagnostics after selection. They do not alter rankings. Space-bearing tokens are printed with `repr` so leading/trailing spaces are visible. The handwriting and custom linguistic vocabularies use letter pairs; original TVA admits all adjacent pairs.


In [5]:
counts = Counter(
    text[i:i + 2] for text in unique_labels for i in range(len(text) - 1)
)
original_pairs = set(selected_bigrams)
handwriting_pairs = {t for t in json.loads(HANDWRITING.read_bytes())['vocab'] if len(t) == 2}
custom_pairs = {t for t in json.loads(CUSTOM_LINGUISTIC.read_bytes())['vocab'] if len(t) == 2}
pair_sets = {'original_tva': original_pairs, 'handwriting': handwriting_pairs, 'custom_linguistic': custom_pairs}
for name, pairs in pair_sets.items():
    require(len(pairs) == 145, f'{name} bigram count mismatch')
nonletter_pairs = [t for t in selected_bigrams if any(c not in string.ascii_letters for c in t)]
vocabulary_comparison = {
    'candidate_pair_count': len(counts),
    'selected_ascii_letter_pairs': 145 - len(nonletter_pairs),
    'selected_nonletter_pairs': nonletter_pairs,
    'selected_space_pairs': [t for t in selected_bigrams if ' ' in t],
    'selected_punctuation_pairs': [t for t in selected_bigrams if any(c in string.punctuation for c in t)],
    'selected_digit_pairs': [t for t in selected_bigrams if any(c in string.digits for c in t)],
    'pairwise_overlap': {
        f'{a} vs {b}': len(pair_sets[a] & pair_sets[b])
        for a, b in combinations(pair_sets, 2)
    },
    'selected_bigrams': [
        {'rank': rank, 'token': token, 'unique_label_occurrence_count': counts[token]}
        for rank, token in enumerate(selected_bigrams, start=1)
    ],
    'selection_boundary_count': counts[selected_bigrams[-1]],
    'pairs_tied_at_selection_boundary': sorted(t for t, count in counts.items() if count == counts[selected_bigrams[-1]]),
}
print(json.dumps({k: v for k, v in vocabulary_comparison.items() if k != 'selected_bigrams'}, indent=2))
for row in vocabulary_comparison['selected_bigrams']:
    print(f'{row["rank"]:3d}  {row["token"]!r:6s}  {row["unique_label_occurrence_count"]}')


{
  "candidate_pair_count": 1397,
  "selected_ascii_letter_pairs": 109,
  "selected_nonletter_pairs": [
    "e ",
    " t",
    "s ",
    " a",
    "t ",
    "d ",
    "n ",
    " o",
    " w",
    " i",
    "r ",
    "y ",
    " s",
    ", ",
    " h",
    " b",
    "o ",
    ". ",
    "f ",
    " f",
    " c",
    " p",
    " m",
    "a ",
    "l ",
    "g ",
    "h ",
    " d",
    " n",
    " e",
    " r",
    " l",
    " M",
    " g",
    "r.",
    " u"
  ],
  "selected_space_pairs": [
    "e ",
    " t",
    "s ",
    " a",
    "t ",
    "d ",
    "n ",
    " o",
    " w",
    " i",
    "r ",
    "y ",
    " s",
    ", ",
    " h",
    " b",
    "o ",
    ". ",
    "f ",
    " f",
    " c",
    " p",
    " m",
    "a ",
    "l ",
    "g ",
    "h ",
    " d",
    " n",
    " e",
    " r",
    " l",
    " M",
    " g",
    " u"
  ],
  "selected_punctuation_pairs": [
    ", ",
    ". ",
    "r."
  ],
  "selected_digit_pairs": [],
  "pairwise_overlap": {
    "original_tva vs handwri

## 6. Audit all ED labels and compare greedy segmentation

This cell requires both local ED archives. It authenticates them with the existing audit reader, checks exact reconstruction and blank exclusion for all three tokenizers, and compares **token strings**, since vocabulary IDs differ. Both archives contain the same 2,550 records; totals per archive are reported separately to avoid treating duplicated records as independent evidence. The supplied folds are retained in per-fold summaries. Compression and segmentation differences are descriptive, not recognition CER/WER.


In [6]:
from audit_ed_handwriting_bigram import read_archive, ARCHIVE_SHA256
import zipfile

tokenizers = {'original_tva': tokenizer_bigram}
for name, kind, artifact in [
    ('handwriting', 'handwriting_bigram_greedy', HANDWRITING),
    ('custom_linguistic', 'linguistic_bigram_greedy', CUSTOM_LINGUISTIC),
]:
    tokenizer = get_tokenizer(kind)
    tokenizer.load(str(artifact))
    require(tokenizer.size == 224 and tokenizer.vocab.get('') == 0, f'{name} dimensions mismatch')
    require({t for t in tokenizer.vocab if len(t) == 1} == set(ED_ALPHABET), f'{name} alphabet mismatch')
    tokenizers[name] = tokenizer


def token_strings(tokenizer, text):
    ids = tokenizer.encode(text)
    require(bool(ids) and 0 not in ids, 'Empty target or CTC blank emitted')
    require(tokenizer.decode(ids) == text, 'Exact ED label reconstruction failed')
    return [tokenizer.decode([index]) for index in ids]


archive_records = {}
segmentation = {}
examples = []
for stem in ARCHIVE_SHA256:
    payload = read_archive(ED_DIRECTORY / f'{stem}.zip', stem)
    rows = payload['annotations']
    records = {row['id']: row['label'] for row in rows}
    require(len(records) == len(rows) == 2550, f'{stem}: expected 2,550 unique sample IDs')
    archive_records[stem] = records
    parts = {name: {} for name in tokenizers}
    for sample_id, text in records.items():
        for name, tokenizer in tokenizers.items():
            parts[name][sample_id] = token_strings(tokenizer, text)
        if stem == 'gold_wd' and len(examples) < 5 and parts['original_tva'][sample_id] != parts['custom_linguistic'][sample_id]:
            examples.append({'id': sample_id, 'label': text, 'tokens': {name: parts[name][sample_id] for name in tokenizers}})
    characters = sum(len(text) for text in records.values())
    summary = {}
    for name in tokenizers:
        total = sum(len(tokens) for tokens in parts[name].values())
        summary[name] = {
            'records': len(records), 'characters': characters, 'encoded_tokens': total,
            'mean_tokens_per_record': total / len(records),
            'token_reduction_vs_characters': 1 - total / characters,
            'bigram_tokens_used': sum(len(t) == 2 for tokens in parts[name].values() for t in tokens),
            'round_trip_failures': 0, 'blank_ids_emitted': 0,
        }
    differences = {
        f'{a} vs {b}': sum(parts[a][i] != parts[b][i] for i in records)
        for a, b in combinations(tokenizers, 2)
    }
    with zipfile.ZipFile(ED_DIRECTORY / f'{stem}.zip') as archive:
        split_payloads = {
            split: json.loads(archive.read(f'{stem}/{split}.json'))
            for split in ('train', 'val')
        }
    folds = {}
    for split, split_payload in split_payloads.items():
        require(split_payload['info']['num_fold'] == 5, 'Expected five ED folds')
        require(set(split_payload['annotations']) == {str(i) for i in range(5)}, 'Invalid ED fold keys')
        for fold_id, fold_rows in split_payload['annotations'].items():
            ids = [row['id'] for row in fold_rows]
            require(len(ids) == len(set(ids)) and set(ids) <= set(records), 'Invalid fold IDs')
            require(all(row['label'] == records[row['id']] for row in fold_rows), 'Fold label mismatch')
            folds.setdefault(fold_id, {})[split] = {
                'records': len(ids),
                'encoded_tokens': {name: sum(len(parts[name][i]) for i in ids) for name in tokenizers},
            }
    for fold_id in folds:
        train_ids = {row['id'] for row in split_payloads['train']['annotations'][fold_id]}
        val_ids = {row['id'] for row in split_payloads['val']['annotations'][fold_id]}
        require(not train_ids & val_ids and train_ids | val_ids == set(records), 'Invalid ED fold partition')
    segmentation[stem] = {'conditions': summary, 'different_segmentations': differences, 'folds': folds}
require(archive_records['gold_wd'] == archive_records['gold_wi'], 'WD/WI records differ')
audit_report = {
    'schema_version': 'tva.iam-original-bigram-comparison.v1',
    'tokenizer_sha256': sha256(OUTPUT.read_bytes()),
    'comparator_sha256': {'handwriting': sha256(HANDWRITING.read_bytes()), 'custom_linguistic': sha256(CUSTOM_LINGUISTIC.read_bytes())},
    'archive_sha256': dict(ARCHIVE_SHA256),
    'unique_ed_records': 2550, 'ed_labels_used_for_vocabulary_selection': False,
    'vocabulary_comparison': vocabulary_comparison, 'segmentation': segmentation,
    'examples': examples,
}
save_frozen(AUDIT_OUTPUT, json_bytes(audit_report))
for stem, report in segmentation.items():
    print(stem, json.dumps({k: v for k, v in report.items() if k != 'folds'}, indent=2))
for example in examples:
    print(json.dumps(example, ensure_ascii=False, indent=2))
print('Audit saved:', AUDIT_OUTPUT)


gold_wd {
  "conditions": {
    "original_tva": {
      "records": 2550,
      "characters": 106087,
      "encoded_tokens": 70766,
      "mean_tokens_per_record": 27.751372549019607,
      "token_reduction_vs_characters": 0.3329437160066737,
      "bigram_tokens_used": 35321,
      "round_trip_failures": 0,
      "blank_ids_emitted": 0
    },
    "handwriting": {
      "records": 2550,
      "characters": 106087,
      "encoded_tokens": 85153,
      "mean_tokens_per_record": 33.39333333333333,
      "token_reduction_vs_characters": 0.19732860765220994,
      "bigram_tokens_used": 20934,
      "round_trip_failures": 0,
      "blank_ids_emitted": 0
    },
    "custom_linguistic": {
      "records": 2550,
      "characters": 106087,
      "encoded_tokens": 79078,
      "mean_tokens_per_record": 31.010980392156863,
      "token_reduction_vs_characters": 0.25459292844552117,
      "bigram_tokens_used": 27009,
      "round_trip_failures": 0,
      "blank_ids_emitted": 0
    }
  },
  "differ

## 7. Recognition training is a separate experiment

Use the frozen artifact with these config fields:

```yaml
tokenizer: bigram
dir_tokenizer: artifacts/tokenizers/ed_iam_tva_original_bigram_greedy_v1.json
```

Ready-to-run configs are now provided at /home/artellisys/TVA/configs/thesis/ed/bigram_tva_original_wd.yaml and /home/artellisys/TVA/configs/thesis/ed/bigram_tva_original_wi.yaml. They use separate result directories and batch 32. Use the matrix condition names `tva_original_wd` and `tva_original_wi` and include fold 0 explicitly.

Match the saved ED study protocol (batch size **32**, seed 42, 300 epochs, BLConv-B + BiLSTM-B, seven input channels, standard augmentation, no concatenation). Superseded batch-64 and custom linguistic templates have been removed; saved historical run configs retain their exact settings. Recognition trains on ED training signals/labels; this single IAM tokenizer remains fixed across WD/WI folds.

Report this condition separately from the custom letter-frequency comparator. It changes both complete-label deduplication and candidate eligibility. The handwriting vocabulary's minimum-count 20 and connected-rate 0.5 thresholds remain provisional. Neither vocabulary overlap nor shorter ED targets demonstrate improved recognition. The supervisor's OnHW-trained transfer result remains incomparable until its exact artifact, config, and scores are available.
